# Cómputo del modelo global

El servidor combina los modelos locales en un único modelo global con la misma arquitectura (`CaffeLeNet.py`). Se comparan tres formas de computarlo:

1. **FedAvg** (McMahan et al., 2017): promedio de los pesos ponderado por el número de ejemplos de cada cliente (sección 3).
2. **Matched averaging** (Wang et al., 2020; Ainsworth et al., 2023): alinea las neuronas de los modelos locales antes de promediarlos (sección 4).
3. **FedAdam** (Reddi et al., 2021): el servidor aplica Adam al cambio promedio de los pesos a lo largo de varias rondas de comunicación (sección 5).

Cada método se compara contra FedAvg en las condiciones para las que fue diseñado, con el mismo presupuesto de entrenamiento local (5 épocas por cliente).

In [ ]:
import glob
import json
import os

import numpy as np
import tensorflow as tf
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from CaffeLeNet import build

SEED = 42
EPOCHS = 5        # mismos valores que en local_training.ipynb,
BATCH_SIZE = 64   # para que todas las comparaciones usen el mismo presupuesto
VAL_SIZE = 0.1
DATA_DIR = "./private/data"  # solo para simular las rondas de la sección 5

### 1. Carga de los modelos locales

`local_training.ipynb` produce dos conjuntos de modelos locales:
- `lmodel_{k}`: todos los clientes parten del mismo modelo inicial $w_0$.
- `lmodel_indep_{k}`: cada cliente parte de una inicialización aleatoria propia.

In [ ]:
def load_local_models(prefix):
    """Carga los modelos locales con el prefijo dado y su número de ejemplos n_k."""
    paths = sorted(glob.glob(f"models/{prefix}_[0-9]*.keras"))
    assert paths, f"No se encontraron modelos models/{prefix}_k.keras"
    models, sizes = [], []
    for path in paths:
        models.append(tf.keras.models.load_model(path))
        with open(path.replace(".keras", ".json"), encoding="utf-8") as f:
            sizes.append(json.load(f)["n_samples"])
    return paths, models, sizes


# Todos los modelos locales deben tener la arquitectura del modelo global.
expected_shapes = [w.shape for w in build.build_it().get_weights()]
local_sets = {}
for prefix in ("lmodel", "lmodel_indep"):
    paths, models, sizes = load_local_models(prefix)
    for path, model in zip(paths, models):
        assert [w.shape for w in model.get_weights()] == expected_shapes, f"{path}: arquitectura distinta"
    local_sets[prefix] = (paths, models, sizes)
    print(f"{prefix}: {[os.path.basename(p) for p in paths]}, n_k = {sizes}")

### 2. Conjunto de prueba

In [ ]:
_, (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_test = np.expand_dims(x_test / 255.0, -1).astype("float32")


def accuracy(model):
    y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
    return float(np.mean(y_pred == y_test)), y_pred


def evaluate(model, name):
    """Calcula la exactitud y guarda el classification report en reports/{name}.txt."""
    acc, y_pred = accuracy(model)
    report = classification_report(y_test, y_pred, digits=4)
    with open(f"reports/{name}.txt", "w", encoding="utf-8") as f:
        f.write(report)
    print(f"Classification report de {name}:\n{report}")
    return acc


def aggregate(weights, name):
    """Construye un modelo global con los pesos dados, lo evalúa y lo guarda."""
    model = build.build_it(name=name)
    model.set_weights(weights)
    acc = evaluate(model, name)
    model.save(f"models/{name}.keras")
    return acc

### 3. FedAvg

FedAvg (McMahan et al., 2017) promedia, parámetro por parámetro, los pesos de los $K$ modelos locales, ponderando cada uno por la fracción de datos con que se entrenó:

$$w_{\text{global}} = \sum_{k=1}^{K} \frac{n_k}{n}\, w_k, \qquad n = \sum_{k=1}^{K} n_k.$$

Como las particiones tienen el mismo tamaño, el resultado coincide con el promedio simple; la ponderación se implementa para que el método sea correcto con particiones de cualquier tamaño. FedAvg supone que todos los clientes parten del mismo modelo inicial $w_0$, por lo que el resultado principal se calcula con los modelos `lmodel_{k}`.

In [ ]:
def fedavg(client_weights, client_sizes):
    """Promedio de los pesos de los clientes, ponderado por n_k / n."""
    coefs = np.asarray(client_sizes, dtype=np.float64)
    coefs /= coefs.sum()
    return [
        sum(c * layer for c, layer in zip(coefs, layers)).astype(np.float32)
        for layers in zip(*client_weights)
    ]


weights_shared = [m.get_weights() for m in local_sets["lmodel"][1]]
sizes_shared = local_sets["lmodel"][2]
acc_fedavg = aggregate(fedavg(weights_shared, sizes_shared), "global_fedavg")

### 4. Matched averaging

Las neuronas ocultas de una red no tienen un orden fijo: permutar las unidades de una capa, y de forma consistente las entradas de la capa siguiente, produce exactamente la misma función. Cuando los clientes parten de inicializaciones distintas, la unidad $j$ de un modelo local no tiene por qué corresponder a la unidad $j$ de otro, y FedAvg promedia unidades que aprendieron rasgos distintos.

*Matched averaging* alinea cada modelo local con un modelo de referencia (el del cliente 0) antes de aplicar FedAvg. Implementamos una versión simplificada del *weight matching* de Ainsworth et al. (2023), inspirado en FedMA (Wang et al., 2020): capa por capa, se busca la permutación $\pi$ de las unidades del cliente que maximiza $\sum_j \langle a_j, b_{\pi(j)} \rangle$, donde $a_j$ y $b_j$ son los pesos de entrada (con el sesgo) de la unidad $j$ en la referencia y en el cliente. Es un problema de asignación que se resuelve con el algoritmo húngaro (`linear_sum_assignment`). La permutación de una capa se aplica también a las entradas de la capa siguiente; entre `pool2` e `ip1`, cada canal corresponde a 16 entradas (4×4) de `Flatten`. La capa de salida no se permuta, porque sus unidades son las clases.

Se evalúa en el escenario para el que fue diseñado, modelos locales con inicialización independiente (`lmodel_indep_{k}`), comparado con FedAvg sobre los mismos modelos. Como control, también se aplica a los modelos con $w_0$ compartido.

In [ ]:
def match_to_reference(reference, weights):
    """Permuta las unidades ocultas de `weights` para alinearlas con `reference`.

    Ambos argumentos son listas [kernel, sesgo, ...] de las capas conv1, conv2,
    ip1 e ip2. Devuelve los pesos permutados; la función del modelo no cambia.
    """
    weights = [w.copy() for w in weights]
    n_layers = len(reference) // 2
    perm = None
    for layer in range(n_layers):
        kernel, bias = weights[2 * layer], weights[2 * layer + 1]

        # Aplicar la permutación de la capa anterior a las entradas de esta capa.
        if perm is not None:
            if kernel.ndim == 4:  # conv -> conv: eje de canales de entrada
                kernel = kernel[:, :, perm, :]
            else:  # conv -> Flatten -> dense (o dense -> dense, con un solo bloque)
                channels = len(perm)
                blocks = kernel.shape[0] // channels
                rows = (np.arange(blocks)[:, None] * channels + perm[None, :]).ravel()
                kernel = kernel[rows, :]

        # Alinear las unidades de salida de esta capa con las de la referencia.
        if layer < n_layers - 1:
            ref_kernel, ref_bias = reference[2 * layer], reference[2 * layer + 1]
            a = np.vstack([ref_kernel.reshape(-1, ref_kernel.shape[-1]), ref_bias])
            b = np.vstack([kernel.reshape(-1, kernel.shape[-1]), bias])
            _, perm = linear_sum_assignment(a.T @ b, maximize=True)
            kernel, bias = kernel[..., perm], bias[perm]

        weights[2 * layer], weights[2 * layer + 1] = kernel, bias
    return weights


def matched_average(client_weights, client_sizes):
    """Alinea todos los modelos con el del cliente 0 y aplica FedAvg."""
    reference = client_weights[0]
    aligned = [reference] + [match_to_reference(reference, w) for w in client_weights[1:]]
    return fedavg(aligned, client_sizes), aligned

In [ ]:
_, models_indep, sizes_indep = local_sets["lmodel_indep"]
weights_indep = [m.get_weights() for m in models_indep]
matched_weights, aligned = matched_average(weights_indep, sizes_indep)

# Verificación: la permutación no cambia las predicciones del modelo alineado.
check = build.build_it()
check.set_weights(aligned[1])
assert np.allclose(
    check.predict(x_test[:512], verbose=0), models_indep[1].predict(x_test[:512], verbose=0), atol=1e-4
), "La permutación cambió la función del modelo"

acc_fedavg_indep = aggregate(fedavg(weights_indep, sizes_indep), "global_fedavg_indep")
acc_matched_indep = aggregate(matched_weights, "global_matched_indep")

# Control: matched averaging sobre modelos que ya comparten w0.
acc_matched_shared = aggregate(matched_average(weights_shared, sizes_shared)[0], "global_matched")

### 5. FedAdam (varias rondas de comunicación)

FedAvg y FedAdam son casos de FedOpt (Reddi et al., 2021). En cada ronda $t$, el servidor envía el modelo global $x_t$ a los clientes; cada cliente entrena una época con sus datos y devuelve sus pesos $w_k$. El servidor calcula el cambio promedio $\Delta_t = \sum_k \frac{n_k}{n}(w_k - x_t)$ y lo usa como pseudo-gradiente:

- **FedAvg:** $x_{t+1} = x_t + \Delta_t$.
- **FedAdam:** $m_t = \beta_1 m_{t-1} + (1-\beta_1)\Delta_t$, $\quad v_t = \beta_2 v_{t-1} + (1-\beta_2)\Delta_t^2$, $\quad x_{t+1} = x_t + \eta\, \dfrac{m_t}{\sqrt{v_t} + \tau}$.

FedAdam acumula momentum entre rondas y adapta el tamaño del paso de cada parámetro. Se usan $\beta_1 = 0.9$, $\beta_2 = 0.99$ y $\tau = 10^{-3}$, valores de Reddi et al. (2021).

Para que la comparación sea justa, ambos métodos parten del mismo $w_0$ y usan `ROUNDS = 5` rondas de una época local: el mismo presupuesto de 5 épocas por cliente que los modelos locales. La tasa de aprendizaje del servidor $\eta$ se elige con la exactitud del modelo global en los conjuntos de validación de los clientes (evaluación federada), sin usar el conjunto de prueba. La comparación final se repite con varias semillas.

Las rondas se simulan en un solo proceso: `local_update` representa el entrenamiento que cada cliente haría en su computadora en cada ronda.

In [ ]:
ROUNDS = 5
SEEDS = [0, 1, 2]
SERVER_LRS = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
BETA1, BETA2, TAU = 0.9, 0.99, 1e-3


def load_client(client_id):
    """Carga la partición privada de un cliente (igual que en local_training.ipynb)."""
    with np.load(os.path.join(DATA_DIR, f"client_{client_id}.npz")) as data:
        x = np.expand_dims(data["x"] / 255.0, -1).astype("float32")
        y = data["y"]
    return x, y


# Misma división entrenamiento/validación que en local_training.ipynb.
clients = []
for client_id in range(len(local_sets["lmodel"][0])):
    x, y = load_client(client_id)
    clients.append(train_test_split(x, y, test_size=VAL_SIZE, stratify=y, random_state=SEED))
client_sizes = [len(y_tr) for _, _, y_tr, _ in clients]


def local_update(global_weights, x, y, seed):
    """Una ronda en un cliente: parte del modelo global y entrena una época."""
    tf.keras.utils.set_random_seed(seed)
    model = build.build_it()
    model.set_weights(global_weights)
    model.fit(x, y, epochs=1, batch_size=BATCH_SIZE, verbose=0)
    return model.get_weights()


def fedavg_step(x, delta, state):
    return [xi + d for xi, d in zip(x, delta)], state


def make_fedadam_step(lr):
    def fedadam_step(x, delta, state):
        if state is None:
            state = ([np.zeros_like(d) for d in delta], [np.full_like(d, TAU**2) for d in delta])
        m, v = state
        m = [BETA1 * mi + (1 - BETA1) * d for mi, d in zip(m, delta)]
        v = [BETA2 * vi + (1 - BETA2) * d**2 for vi, d in zip(v, delta)]
        x = [xi + lr * mi / (np.sqrt(vi) + TAU) for xi, mi, vi in zip(x, m, v)]
        return x, (m, v)
    return fedadam_step


def initial_weights(seed):
    tf.keras.utils.set_random_seed(SEED + seed)
    return build.build_it().get_weights()


def run_rounds(server_step, w0, seed):
    """Simula ROUNDS rondas de comunicación y devuelve los pesos globales finales."""
    x = [w.astype(np.float64) for w in w0]
    state = None
    for t in range(ROUNDS):
        sent = [w.astype(np.float32) for w in x]
        client_weights = [
            local_update(sent, x_tr, y_tr, seed=seed * 1000 + t * 10 + k)
            for k, (x_tr, _, y_tr, _) in enumerate(clients)
        ]
        delta = [avg - xi for avg, xi in zip(fedavg(client_weights, client_sizes), x)]
        x, state = server_step(x, delta, state)
    return [w.astype(np.float32) for w in x]


def federated_val_accuracy(weights):
    """Exactitud del modelo global sobre la unión de los conjuntos de validación de los clientes."""
    model = build.build_it()
    model.set_weights(weights)
    correct = sum(
        int(np.sum(np.argmax(model.predict(x_val, verbose=0), axis=1) == y_val))
        for _, x_val, _, y_val in clients
    )
    return correct / sum(len(y_val) for _, _, _, y_val in clients)

**Selección de $\eta$** con la primera semilla y evaluación federada:

In [ ]:
w0 = initial_weights(SEEDS[0])
val_acc = {}
for lr in SERVER_LRS:
    val_acc[lr] = federated_val_accuracy(run_rounds(make_fedadam_step(lr), w0, SEEDS[0]))
    print(f"eta = {lr:g}: exactitud en validación federada = {val_acc[lr]:.4f}")
best_lr = max(val_acc, key=val_acc.get)
print(f"eta seleccionada: {best_lr:g}")

**Comparación final** en el conjunto de prueba, con las mismas semillas para ambos métodos:

In [ ]:
acc_rounds = {"fedavg_rondas": [], "fedadam_rondas": []}
for seed in SEEDS:
    w0 = initial_weights(seed)
    for name, step in (("fedavg_rondas", fedavg_step), ("fedadam_rondas", make_fedadam_step(best_lr))):
        weights = run_rounds(step, w0, seed)
        if seed == SEEDS[0]:  # se guardan el modelo y el reporte de la primera semilla
            acc_rounds[name].append(aggregate(weights, f"global_{name}"))
        else:
            model = build.build_it()
            model.set_weights(weights)
            acc_rounds[name].append(accuracy(model)[0])

for name, accs in acc_rounds.items():
    print(f"{name}: {np.mean(accs):.4f} ± {np.std(accs, ddof=1):.4f}  (semillas: {[round(a, 4) for a in accs]})")

### 6. Referencias de comparación

- **Modelos locales**: exactitud de cada modelo local en el conjunto de prueba (sus reportes completos están en `reports/`).
- **Modelo centralizado**: el mismo modelo entrenado con los 60,000 ejemplos de entrenamiento en un solo dispositivo, con las mismas épocas y tamaño de lote. Es una cota superior de referencia, no un escenario federado: requiere reunir todos los datos en un solo lugar.

In [ ]:
local_acc = {
    os.path.basename(path).removesuffix(".keras"): accuracy(model)[0]
    for prefix in ("lmodel", "lmodel_indep")
    for path, model in zip(local_sets[prefix][0], local_sets[prefix][1])
}

(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = np.expand_dims(x_train / 255.0, -1).astype("float32")

tf.keras.utils.set_random_seed(SEED)
central_model = build.build_it(name="centralizado")
central_model.fit(x_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)
acc_central = evaluate(central_model, "centralizado")

### 7. Resumen de resultados

In [ ]:
def mean_local(prefix):
    return [float(np.mean([acc for name, acc in local_acc.items() if name.rsplit("_", 1)[0] == prefix]))]


# (escenario, método, exactitudes en prueba)
results = [
    ("w0 compartido, 1 ronda", "promedio de modelos locales", mean_local("lmodel")),
    ("w0 compartido, 1 ronda", "FedAvg", [acc_fedavg]),
    ("w0 compartido, 1 ronda", "Matched averaging (control)", [acc_matched_shared]),
    ("inicialización independiente, 1 ronda", "promedio de modelos locales", mean_local("lmodel_indep")),
    ("inicialización independiente, 1 ronda", "FedAvg", [acc_fedavg_indep]),
    ("inicialización independiente, 1 ronda", "Matched averaging", [acc_matched_indep]),
    (f"w0 compartido, {ROUNDS} rondas", "FedAvg", acc_rounds["fedavg_rondas"]),
    (f"w0 compartido, {ROUNDS} rondas", f"FedAdam (eta = {best_lr:g})", acc_rounds["fedadam_rondas"]),
    ("datos centralizados", "modelo centralizado", [acc_central]),
]

with open("reports/resultados.csv", "w", encoding="utf-8") as f:
    f.write("escenario,metodo,exactitud_media,desv_est,n_semillas\n")
    for scenario, method, accs in results:
        sd = f"{np.std(accs, ddof=1):.4f}" if len(accs) > 1 else ""
        f.write(f"{scenario},{method},{np.mean(accs):.4f},{sd},{len(accs)}\n")

print(f"{'Escenario':<40}{'Método':<32}Exactitud (prueba)")
for scenario, method, accs in results:
    sd = f" ± {np.std(accs, ddof=1):.4f} (n = {len(accs)})" if len(accs) > 1 else ""
    print(f"{scenario:<40}{method:<32}{np.mean(accs):.4f}{sd}")